# ECMWF IFS HRES — Download

**Model:** ECMWF Integrated Forecasting System (IFS) — High-Resolution Ensemble System (HRES)  
**Type:** Physics-based deterministic  
**Cycles:** 00Z and 12Z  
**Range:** 0–360h (steps 0–144h at 3-hourly, 144–360h at 6-hourly)

## Files produced

```
data/YYYY-MM-DD_HHZ/
    msl_f024.grib2           T+24h verification snapshots
    sfc_f024.grib2
    pl_f024.grib2
    wave_f024.grib2
    sfc_f000-f120_3h.grib2   Group 1 — surface (3-hourly, 0–120h)
    pl_f000-f120_12h.grib2   Group 2 — upper air (12-hourly, 0–120h)
    wave_f000-f120_3h.grib2  Group 3 — wave (3-hourly, 0–120h)
    tc_tracks_hres.bufr      Group 4 — TC tracks (when active)
```

In [ ]:
from ecmwf.opendata import Client
from pathlib import Path
import xarray as xr
import cfgrib

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
client = Client(source="ecmwf", model="ifs")
latest_run  = client.latest()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)

print("Latest run  :", latest_run)
print("Cycle (UTC) :", latest_time)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── Verification snapshots — T+24h ────────────────────────────────────────────
client.retrieve(time=latest_time, step=24, type="fc",
    param="msl", target=str(output_dir / "msl_f024.grib2"))

client.retrieve(time=latest_time, step=24, type="fc",
    param=["2t", "msl", "10u", "10v", "skt", "sstk"],
    target=str(output_dir / "sfc_f024.grib2"))

client.retrieve(time=latest_time, step=24, type="fc",
    param=["gh", "t", "u", "v"], levtype="pl", levelist=[850, 700, 500, 300, 250],
    target=str(output_dir / "pl_f024.grib2"))

client.retrieve(time=latest_time, step=24, type="fc",
    stream="wave", param=["swh", "mwd", "pp1d"],
    target=str(output_dir / "wave_f024.grib2"))

print("Snapshots saved.")

In [ ]:
# ── Group 1: Surface fields — 3-hourly, 0–120h ───────────────────────────────
steps_3h = list(range(0, 121, 3))

SFC = {}
SFC["thermo"]      = ["2t", "2d", "skt", "sstk"]
SFC["wind"]        = ["10u", "10v", "10fg", "100u", "100v"]
SFC["pressure"]    = ["msl", "sp"]
SFC["moisture"]    = ["tp", "cp", "tcc", "lcc", "mcc", "hcc", "tcwv", "ro"]
SFC["energy"]      = ["ssrd", "strd", "ttr", "blh"]
SFC["instability"] = ["mucape", "mucin"]

client.retrieve(
    time=latest_time, step=steps_3h, type="fc",
    param=[p for grp in SFC.values() for p in grp],
    target=str(output_dir / "sfc_f000-f120_3h.grib2"),
)
print("Group 1 saved: sfc_f000-f120_3h.grib2")
print(f"  Groups: { {k: len(v) for k, v in SFC.items()} }")

In [ ]:
# ── Group 2: Upper-air — 12-hourly, 0–120h ───────────────────────────────────
PL = {}
PL["thermo"]   = ["t", "gh"]
PL["wind"]     = ["u", "v", "w"]
PL["moisture"] = ["q", "r"]
PL["dynamics"] = ["vo", "d"]

client.retrieve(
    time=latest_time, step=list(range(0, 121, 12)), type="fc",
    param=[p for grp in PL.values() for p in grp],
    levtype="pl", levelist=[1000, 925, 850, 700, 600, 500, 400, 300, 250, 200],
    target=str(output_dir / "pl_f000-f120_12h.grib2"),
)
print("Group 2 saved: pl_f000-f120_12h.grib2")
print(f"  Groups: { {k: len(v) for k, v in PL.items()} }")

In [ ]:
# ── Group 3: Wave — 3-hourly, 0–120h ─────────────────────────────────────────
client.retrieve(
    time=latest_time, step=steps_3h, type="fc",
    stream="wave", param=["swh", "mwd", "mwp", "pp1d"],
    target=str(output_dir / "wave_f000-f120_3h.grib2"),
)
print("Group 3 saved: wave_f000-f120_3h.grib2")

In [ ]:
# ── Group 4: TC tracks — when active ────────────────────────────────────────
try:
    client.retrieve(time=latest_time, stream="oper", type="tf", step=240,
        target=str(output_dir / "tc_tracks_hres.bufr"))
    print("Group 4 saved: tc_tracks_hres.bufr")
except ValueError:
    print("Group 4: no active TCs this run")